# LCE Review Pipeline — 02 · AI Enrich (Silver → Gold)

Reads `reviews_silver` and writes `reviews_gold` — the serving table for the Guest
Sentiment app / dashboards.

**One `ai_query` call per review** extracts everything in a single structured pass:
classification, sentiment, per-aspect ratings, products mentioned, **and the linked
`product_issues` pairs** (each food-quality issue tied to the product it actually
describes). Folding `product_issues` into the same schema means **no second FM pass** —
half the cost and latency of running it as a separate query.

| Layer | Table          | Contents                                                        |
|-------|----------------|-----------------------------------------------------------------|
| Gold  | `reviews_gold` | Silver + AI classification/sentiment/ratings/product + `product_issues` |

> **Env:** `catalog`/`schema` widgets (test default `jdub_demo.little_caesars`; Job
> overrides to prod `ioc_sandbox.ai_strategy`). `stores_table` optionally joins a store
> dimension for `location_name`; leave blank to fall back to `locationId`.

In [ ]:
# --- Parameters (widgets) ---------------------------------------------------
dbutils.widgets.text("catalog", "jdub_demo", "Catalog")
dbutils.widgets.text("schema", "little_caesars", "Schema")
# Optional store dimension for location_name. Prod: ioc_sandbox.ai_strategy.worst_performing_stores
# Leave blank in test if no such table exists -> location_name falls back to the locationId.
dbutils.widgets.text("stores_table", "", "Stores dim table (FQN, optional)")
dbutils.widgets.text("stores_id_col", "Location ID", "Stores: location id column")
dbutils.widgets.text("model", "databricks-gpt-oss-20b", "FM endpoint")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA  = dbutils.widgets.get("schema").strip()
STORES  = dbutils.widgets.get("stores_table").strip()
STORES_ID = dbutils.widgets.get("stores_id_col").strip()
MODEL   = dbutils.widgets.get("model").strip()
FQ = lambda t: f"{CATALOG}.{SCHEMA}.{t}"

SILVER = FQ("reviews_silver")
GOLD   = FQ("reviews_gold")
print(f"{SILVER}  ->  {GOLD}")
print(f"model: {MODEL}   stores join: {STORES or '(none — location_name = locationId)'}")

## Gold — single-pass AI enrichment

The `responseFormat` JSON schema returns every field in one call. Notable choices:

- **`product`** is extracted whenever a product is named *anywhere* in the review (not
  gated on the Quality aspect), so "no pepperoni in my order" is captured — and it's
  **enum-constrained** so values don't fragment (`pizza`/`Pizza`/`cheese pizza`).
- **`product_issues`** is the linked pairing: for each product the reviewer complains
  about, the specific food-quality issue *with that product*. This is what powers the
  Theme Explorer bubble viz, and it's extracted here rather than in a second `ai_query`.
- `failOnError => false` + the `errorMessage IS NULL` filter drops any row the model
  errored on rather than failing the whole job.

In [ ]:
# Build the gold SQL in Python so the store-dimension join is optional (widget-driven).
_stores_join = ""
_location_name = "CAST(b.locationId AS STRING) AS location_name"
if STORES:
    _stores_join = f"LEFT JOIN {STORES} s ON b.locationId = s.`{STORES_ID}`"
    # "City, ST" (matches sentiment_tab_update/sql/gold_add_location_name.sql). The app
    # takes split(location_name, ',')[0] as the store label and the last comma-segment as
    # the State filter, so City/State is the shape it expects. concat_ws skips NULL parts;
    # NULLIF('' -> NULL) lets an unmatched store fall back to the raw locationId.
    _location_name = "COALESCE(NULLIF(concat_ws(', ', s.City, s.State), ''), CAST(b.locationId AS STRING)) AS location_name"

GOLD_SQL = f"""
CREATE OR REPLACE TABLE {GOLD} AS
WITH base AS (
  SELECT
    locationId, date, rating, rating_missing, directoryType, review_text,
    ai_query(
      '{MODEL}',
      concat(
        'You are analyzing a Little Caesars (pizza QSR) customer review. Extract all of the following. ',
        '(1) classification: every applicable experience aspect. ',
        '(2) sentiment: overall. (3) comment: one brief sentence. ',
        '(4) a 1-5 rating for each aspect only if it is actually discussed, else null. ',
        '(5) product: any specific menu products named ANYWHERE in the review (not only in a quality context; ',
        'include items from accuracy/speed complaints like \\'no pepperoni\\' or \\'forgot my breadsticks\\'); null if none. ',
        '(6) product_issues: ONLY for negative reviews, the list of {{product, issue}} pairs where each issue is the ',
        'specific food-quality problem with THAT product. Pair an issue only with the product it actually describes; ',
        'if a product is praised or mentioned in passing, do not include it; empty list if no product-specific food-quality complaint.\\n\\n',
        review_text
      ),
      responseFormat => '{{
        "type": "json_schema",
        "json_schema": {{
          "name": "review_extraction",
          "schema": {{
            "type": "object",
            "properties": {{
              "classification":        {{ "type": "array", "items": {{ "type": "string", "enum": ["Speed", "Cleanliness", "Order Accuracy", "Quality", "Service"] }} }},
              "comment":               {{ "type": "string" }},
              "sentiment":             {{ "type": "string", "enum": ["Positive", "Negative", "Neutral", "Mixed"] }},
              "speed_rating":          {{ "anyOf": [{{"type": "number"}}, {{"type": "null"}}] }},
              "cleanliness_rating":    {{ "anyOf": [{{"type": "number"}}, {{"type": "null"}}] }},
              "order_accuracy_rating": {{ "anyOf": [{{"type": "number"}}, {{"type": "null"}}] }},
              "quality_rating":        {{ "anyOf": [{{"type": "number"}}, {{"type": "null"}}] }},
              "service_rating":        {{ "anyOf": [{{"type": "number"}}, {{"type": "null"}}] }},
              "product":               {{ "anyOf": [{{ "type": "array", "items": {{ "type": "string", "enum": ["Pizza","Crazy Bread","Wings","Crazy Puffs","Deep Dish","Stuffed Crust","Cheese","Sauce","Crust","Pepperoni"] }} }}, {{ "type": "null" }}] }},
              "product_issues":        {{ "type": "array", "items": {{ "type": "object", "properties": {{
                                            "product": {{ "type": "string", "enum": ["Pizza","Crazy Bread","Wings","Crazy Puffs","Deep Dish","Stuffed Crust","Cheese","Sauce","Crust","Pepperoni"] }},
                                            "issue":   {{ "type": "string", "enum": ["Soggy","Cold","Undercooked","Burnt","Greasy","Stale/Dry","Bland","Skimpy toppings","Small portion"] }}
                                          }}, "required": ["product","issue"], "additionalProperties": false }} }}
            }},
            "required": ["classification","comment","sentiment","speed_rating","cleanliness_rating","order_accuracy_rating","quality_rating","service_rating","product","product_issues"],
            "additionalProperties": false
          }},
          "strict": true
        }}
      }}',
      failOnError => false
    ) AS extracted_opinion
  FROM {SILVER}
  WHERE review_text IS NOT NULL
)
SELECT
  b.locationId,
  {_location_name},
  b.date, b.rating, b.rating_missing, b.directoryType, b.review_text,
  from_json(b.extracted_opinion.result:classification, 'ARRAY<STRING>')      AS classification,
  b.extracted_opinion.result:comment::STRING                                 AS comment,
  b.extracted_opinion.result:sentiment::STRING                               AS sentiment,
  b.extracted_opinion.result:speed_rating::DOUBLE                            AS speed_rating,
  b.extracted_opinion.result:cleanliness_rating::DOUBLE                      AS cleanliness_rating,
  b.extracted_opinion.result:order_accuracy_rating::DOUBLE                   AS order_accuracy_rating,
  b.extracted_opinion.result:quality_rating::DOUBLE                          AS quality_rating,
  b.extracted_opinion.result:service_rating::DOUBLE                          AS service_rating,
  from_json(b.extracted_opinion.result:product, 'ARRAY<STRING>')            AS product,
  from_json(b.extracted_opinion.result:product_issues, 'ARRAY<STRUCT<product STRING, issue STRING>>') AS product_issues
FROM base b
{_stores_join}
WHERE b.extracted_opinion.errorMessage IS NULL
"""

print(GOLD_SQL)
spark.sql(GOLD_SQL)
print(f"\nWrote {GOLD}")

## Validate gold

Row count, extraction health, and the distinct `product × issue` pairs the Theme Explorer
will render.

In [ ]:
from pyspark.sql import functions as F

g = spark.table(GOLD)
print(f"gold rows: {g.count()}")

# Sentiment split.
display(g.groupBy("sentiment").count().orderBy(F.desc("count")))

# product_issues health over negatives.
neg = g.filter(F.col("sentiment") == "Negative")
print(f"negatives: {neg.count()}  |  with >=1 product_issue: "
      f"{neg.filter(F.size('product_issues') > 0).count()}")

# Distinct product x issue pairs (what the viz renders).
pairs = (g.select(F.explode("product_issues").alias("pi"))
          .select("pi.product", "pi.issue")
          .groupBy("product", "issue").count().orderBy(F.desc("count")))
display(pairs)

In [ ]:
# Quick spot-check: a few negative reviews with their extracted pairs.
display(
    spark.table(GOLD)
    .filter((F.col("sentiment") == "Negative") & (F.size("product_issues") > 0))
    .select("location_name", "rating", "product_issues", "review_text")
    .limit(10)
)